# 🎓 Train Fake News Detection on Google Colab

**Notebook này:**
1. Clone repo từ GitHub
2. Tải dataset FakeNewsNet (wget)
3. Chuẩn bị data (train/val/test split)
4. Train model với curriculum learning
5. Lưu artifacts

In [9]:
# 🔧 Kiểm tra GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  Bạn nên chuyển Runtime sang GPU để train nhanh hơn.')

CUDA available: True
GPU: Tesla T4


In [10]:
# ===== Cấu hình repo =====
REPO_URL = 'https://github.com/phidinhmanh/fake-news-detection.git'
BRANCH = 'main'
REPO_DIR = '/content/fake-new-detection'

# ===== Cấu hình so sánh =====
MODELS_TO_COMPARE = [
    {'name': 'Baseline (TF-IDF)', 'type': 'baseline'},
    {'name': 'XLM-RoBERTa (LoRA)', 'type': 'transformer', 'model_name': 'xlm-roberta-base'},
    # {'name': 'BERT-Multi (LoRA)', 'type': 'transformer', 'model_name': 'bert-base-multilingual-cased'}
]

EPOCHS_LORA = 3
BATCH_SIZE = 32
USE_CURRICULUM = True

results_comparison = {}

In [11]:
# 📥 Clone repo từ GitHub
import os
import subprocess

if USE_PRIVATE_REPO:
    if not GITHUB_TOKEN:
        raise ValueError('Repo private thì cần GITHUB_TOKEN')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    clone_url = REPO_URL

if os.path.exists(REPO_DIR):
    print('Repo đã tồn tại, pull code mới...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True, capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True, capture_output=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True, capture_output=True)
else:
    print('Đang clone repo...')
    result = subprocess.run(
        ['git', 'clone', '-b', BRANCH, clone_url, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('Clone failed:', result.stderr)
    else:
        print('Clone thành công!')

print('Sẵn sàng tại:', REPO_DIR)

Repo đã tồn tại, pull code mới...
Sẵn sàng tại: /content/fake-new-detection


In [12]:
# 📥 Tải FakeNewsNet Dataset (sử dụng wget)
import os

%cd /content/fake-new-detection

# Tạo thư mục data/raw
!mkdir -p data/raw

# URLs của 4 file CSV từ FakeNewsNet GitHub
FAKENEWSNET_URLS = {
    'politifact_fake.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_fake.csv',
    'politifact_real.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/politifact_real.csv',
    'gossipcop_fake.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_fake.csv',
    'gossipcop_real.csv': 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset/gossipcop_real.csv',
}

print('Đang tải FakeNewsNet dataset...')
for filename, url in FAKENEWSNET_URLS.items():
    filepath = f'data/raw/{filename}'
    if os.path.exists(filepath):
        print(f'  ✓ {filename} đã tồn tại')
    else:
        print(f'  ⏳ Tải {filename}...')
        result = subprocess.run(
            ['wget', '-q', '-O', filepath, url],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  ✓ {filename} tải thành công')
        else:
            print(f'  ✗ Lỗi tải {filename}: {result.stderr}')

/content/fake-new-detection
Đang tải FakeNewsNet dataset...
  ✓ politifact_fake.csv đã tồn tại
  ✓ politifact_real.csv đã tồn tại
  ✓ gossipcop_fake.csv đã tồn tại
  ✓ gossipcop_real.csv đã tồn tại


In [13]:
# 🧹 Chuẩn bị data: Merge và Split
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

print('Đang đọc và merge dataset...')

all_dfs = []
for filename in FAKENEWSNET_URLS.keys():
    filepath = f'data/raw/{filename}'
    df = pd.read_csv(filepath)
    
    # Gắn nhãn source và label
    parts = filename.replace('.csv', '').split('_')
    df['source'] = parts[0]  # politifact / gossipcop
    df['label'] = parts[1]   # fake / real
    df['label_binary'] = (df['label'] == 'fake').astype(int)
    
    all_dfs.append(df)
    print(f'  - {filename}: {len(df)} records')

# Merge all
df_all = pd.concat(all_dfs, ignore_index=True)
print(f'\nTổng cộng: {len(df_all)} records')

# Làm sạch text
df_all['text'] = df_all['title'].fillna('').astype(str)

# Rename columns
df_all = df_all.rename(columns={
    'id': 'news_id',
    'url': 'news_url',
})

# Split data: 80% train, 10% val, 10% test
print('\nĐang split data...')
train_df, temp_df = train_test_split(df_all, test_size=0.2, random_state=42, stratify=df_all['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f'  Train: {len(train_df)} records')
print(f'  Val:   {len(val_df)} records')
print(f'  Test:  {len(test_df)} records')

# Lưu CSV
df_all.to_csv('data/raw/fakenewsnet_clean.csv', index=False, encoding='utf-8')
train_df.to_csv('data/raw/train.csv', index=False, encoding='utf-8')
val_df.to_csv('data/raw/val.csv', index=False, encoding='utf-8')
test_df.to_csv('data/raw/test.csv', index=False, encoding='utf-8')

# Lưu Parquet (cho training nhanh hơn)
os.makedirs('data/datasets/normalized', exist_ok=True)
train_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/train.parquet', index=False)
val_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/val.parquet', index=False)
test_df[['text', 'label', 'label_binary', 'source']].to_parquet('data/datasets/normalized/test.parquet', index=False)

print('\n✅ Data đã chuẩn bị xong!')
print(f'  - data/raw/fakenewsnet_clean.csv')
print(f'  - data/datasets/normalized/train.parquet')
print(f'  - data/datasets/normalized/val.parquet')
print(f'  - data/datasets/normalized/test.parquet')

Đang đọc và merge dataset...
  - politifact_fake.csv: 432 records
  - politifact_real.csv: 624 records
  - gossipcop_fake.csv: 5323 records
  - gossipcop_real.csv: 16817 records

Tổng cộng: 23196 records

Đang split data...
  Train: 18556 records
  Val:   2320 records
  Test:  2320 records

✅ Data đã chuẩn bị xong!
  - data/raw/fakenewsnet_clean.csv
  - data/datasets/normalized/train.parquet
  - data/datasets/normalized/val.parquet
  - data/datasets/normalized/test.parquet


In [ ]:

# 🚀 MODEL 1: Baseline TF-IDF + Logistic Regression
import sys
import os
from pathlib import Path
import pandas as pd

%cd /content/fake-new-detection

from model.baseline_logreg import BaselineLogReg
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print('🚀 Đang training Baseline (TF-IDF + LogReg)...')
train_df = pd.read_csv('data/raw/train.csv')
test_df = pd.read_csv('data/raw/test.csv')

baseline = BaselineLogReg()
baseline.train(train_df)

# Evaluation
y_true = test_df['label']
y_pred = baseline.pipeline.predict(test_df['text'])
y_proba = baseline.pipeline.predict_proba(test_df['text'])[:, 1]

metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred, pos_label='fake'),
    'precision': precision_score(y_true, y_pred, pos_label='fake'),
    'recall': recall_score(y_true, y_pred, pos_label='fake')
}

results_comparison['Baseline (TF-IDF)'] = metrics

print('\n📊 Baseline Results:')
for k, v in metrics.items():
    print(f'  {k.capitalize()}: {v:.4f}')

In [14]:
# 📦 Cài đặt dependencies
%cd /content/fake-new-detection

# Cài dependencies cho module model
!pip -q install -r requirements-model.txt 2>/dev/null || true

# Cài thêm các thư viện cần thiết
!pip -q install pyarrow transformers peft lightning scikit-learn pandas tqdm

print('✅ Dependencies đã cài đặt!')

/content/fake-new-detection
✅ Dependencies đã cài đặt!


In [15]:
# 🚀 MODEL 2: Deep Learning Models (Transformer-based)
import subprocess
import shlex
import os
import json

transformer_models = [m for m in MODELS_TO_COMPARE if m['type'] == 'transformer']

for m in transformer_models:
    print(f'\n' + '='*60)
    print(f'🔥 Training Model: {m["name"]} ({m["model_name"]})')
    print('='*60)
    
    # Chạy script train.py
    cmd = ['python', 'model/train.py', '--batch-size', str(BATCH_SIZE), '--epochs', str(EPOCHS_LORA)]
    if USE_CURRICULUM:
        cmd.append('--curriculum')
    
    # Lưu ý: Cần config.yaml phù hợp cho từng model nếu muốn đổi model_name
    # Hiện tại train.py đọc từ model/config.yaml. Ta sẽ override model name trong command nếu train.py hỗ trợ
    # Nếu không, ta sẽ sửa model/config.yaml tạm thời
    
    import yaml
    with open('model/config.yaml', 'r') as f:
        config = yaml.safe_load(f)
    config['model']['name'] = m['model_name']
    with open('model/config.yaml', 'w') as f:
        yaml.dump(config, f)

    print('Run command:', ' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd)
    
    # Sau khi train xong, ta giả định test results được in ra hoặc lưu lại
    # Trong train.py, nó in test results ra stdout. 
    # Để đơn giản, ở đây ta sẽ parse kết quả từ log nếu cần, hoặc chạy một script evaluate riêng
    # Giả sử ta lấy kết quả từ artifact checkpoints nếu tốn thời gian parse stdout
    
    # Tạm thời gán kết quả giả lập nếu không parse được stdout dễ dàng (vì r.stdout không capture ở notebook cell)
    # Thực tế nên bổ sung train.py để save metrics ra file json
    
    # Đọc kết quả test (giả sử train.py save vào results/metrics.json)
    # Ở đây ta sẽ chạy evaluate script nếu cần
    print(f'✅ Đã train xong {m["name"]}')

Run command: python model/train.py --config model/config.yaml --curriculum --epochs 5 --batch-size 64
Return code: 0


In [16]:
# ✅ Kiểm tra artifacts
from pathlib import Path

artifacts = [
    Path('saved_models/final_model.pt'),
    Path('saved_models/lora_adapter'),
    Path('saved_models/checkpoints'),
]
for p in artifacts:
    status = '✅ OK' if p.exists() else '❌ MISSING'
    print(f'{p}: {status}')

print('\n📁 Top checkpoint files:')
checkpoints_dir = Path('saved_models/checkpoints')
if checkpoints_dir.exists():
    for ckpt in sorted(checkpoints_dir.glob('*.ckpt'))[:5]:
        print(f'  - {ckpt.name}')
else:
    print('  (No checkpoints yet)')

saved_models/final_model.pt: ✅ OK
saved_models/lora_adapter: ✅ OK
saved_models/checkpoints: ✅ OK

📁 Top checkpoint files:
  - fake-news-epoch=01-val_f1=0.8585.ckpt
  - fake-news-epoch=02-val_f1=0.8585.ckpt
  - fake-news-epoch=02-val_f1=0.8628.ckpt
  - fake-news-epoch=03-val_f1=0.8589.ckpt
  - fake-news-epoch=03-val_f1=0.8635.ckpt


In [17]:
# 💾 Optional: Lưu artifact sang Google Drive
SAVE_TO_DRIVE = False
DRIVE_TARGET = '/content/drive/MyDrive/fake-news-models'

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    import os

    drive.mount('/content/drive')
    os.makedirs(DRIVE_TARGET, exist_ok=True)
    shutil.copytree('saved_models', f'{DRIVE_TARGET}/saved_models', dirs_exist_ok=True)
    print('Đã copy saved_models ->', DRIVE_TARGET)
else:
    print('Đặt SAVE_TO_DRIVE = True nếu muốn backup lên Google Drive.')

Đặt SAVE_TO_DRIVE = True nếu muốn backup lên Google Drive.


In [ ]:
# 📦 Optional: Zip artifact để download
!zip -rq saved_models.zip saved_models
print('Đã tạo saved_models.zip')
print('(Download từ file browser bên trái)')
# from google.colab import files
# files.download('saved_models.zip')

# 📊 Final Comparison Summary

So sánh hiệu năng giữa Baseline và các model Deep Learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

if results_comparison:
    df_res = pd.DataFrame(results_comparison).T
    print('📊 Bảng so sánh kết quả (Test Set):')
    display(df_res)

    # Vẽ biểu đồ
    df_res[['accuracy', 'f1']].plot(kind='bar', figsize=(10, 6))
    plt.title('Model Comparison: Accuracy vs F1-Score')
    plt.ylabel('Score')
    plt.ylim(0, 1.0)
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(loc='lower right')
    plt.show()
else:
    print('Chưa có kết quả so sánh.')